# Tutorial 2: Artifacts and Noise in CT, and 3D CT reconstruction
---
Faculty of Mathematics, Vienna, March 25, 2026.

Author: Sonia Foschiatti

---

### Learning Objectives
In this notebook, you will:
1. Understand how artifacts and noise affect CT reconstructions
2. Build a 3D phantom
3. Perform a 3D CT reconstruction and a 3D visualization

---
## Setup and Imports

First, let's import the necessary libraries:

In [ ]:
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
from skimage.data import shepp_logan_phantom
from skimage.transform import radon, iradon, resize, rescale
from skimage.util import random_noise
import io
from matplotlib.widgets import Slider, CheckButtons
from ipywidgets import IntSlider, Image, HBox, Layout, VBox
from PIL import Image as PILImage

import warnings
warnings.filterwarnings('ignore')


print("Libraries imported successfully!")

Let us begin from the artifacts in CT.

## X-ray tomography with limited data

### Sparse full-angle tomography

In many real-world scenarios, there are often restrictions on the number of available projection directions or an angle of view. 

With full-data tomography, the filtered-back-projection type algorithms are still efficient and used nowadays in CT machines. This is no longer true when dealing with **sparse full-angle tomography**. In this case, the projections are taken from full angle but with large uniform angular steps. 

Let's explore how this affects reconstruction quality.

In [ ]:
# Recall the definition of the phantom 
image = shepp_logan_phantom()
image = rescale(image, scale=0.4, mode='reflect', channel_axis=None)

In [ ]:
# Let us define the RMSE
def RMSE(x, y):
    return float(np.sqrt(np.mean((x - y)**2)))

In [ ]:
num_angles = [10, 30, 60, 180]

fig, axes = plt.subplots(2, 4, figsize=(20, 10))

for i, num_proj in enumerate(num_angles):
    
    # Create an array with sparse full-data projections
    theta_limited = np.linspace(0., 180., num_proj, endpoint=False)
    
    # Compute the Radon transform with sparse angles
    sinogram_limited = radon(image, theta=theta_limited)
    
    # Reconstruction step
    reconstruction_limited = iradon(sinogram_limited, theta=theta_limited, 
                                    filter_name='ramp')
    
    # Calculate error
    error_limited = RMSE(image,reconstruction_limited)
    
    # Plot sinogram
    axes[0, i].imshow(sinogram_limited, cmap='gray', aspect='auto')
    axes[0, i].set_title(f'Sinogram: {num_proj} angles', fontsize=12)
    axes[0, i].axis('off')
    
    # Plot reconstruction
    axes[1, i].imshow(reconstruction_limited, cmap='gray')
    axes[1, i].set_title(f'Reconstruction: {num_proj} angles\nRMSE: {error_limited:.4f}', 
                          fontsize=12)
    axes[1, i].axis('off')

plt.suptitle('Sparse Full-Angle Tomography', fontsize=12)
plt.tight_layout()
plt.show()

### Exercise 1:

With the help of the above code, write a function called `RMSE_sparse_angle` that takes as inputs:
- the number of projection angles `num_proj`;
- the original image `image`;
- a filter `filter_name`.

and returns
- the reconstructed image
- the RMSE (to implement it, use the above function) 

Next, using as image the Shepp-Logan phantom introduced above, iterate on the given list `num_angles` using as default filter the `ramp` filter and print the RMSE. (**Optional**: by slightly modifying the above code, plot the reconstructions and in the title add the RMSE value) 

**Question**: Does doubling the number of projections always halve the error?  

**Time**: 10 minutes

In [ ]:
# Write your function here.

In [ ]:
# Write your code here
num_angles = [30, 60, 120, 240]
filter_name = 'ramp'

### Limited-angle tomography

In many applications, the tomographic measurements can be acquired only at a limited angle range, which means that the available angles for the Radon transform $\mathcal{R}[f](t,\theta)$ vary in an interval $[\Phi,\pi-\Phi]$ for $\Phi<\pi/2$. 

This is another example of (severely) **ill-posed** problem. Typical artifacts are **blurring** and **elongation**.

Let us explore this in the following cells.

In [ ]:
# Define the array theta and the sinogram
theta = np.linspace(0, 120, 24, endpoint=True)
sinogram_la = radon(image, theta=theta, circle=False)

# Reconstruct the image using filtered-back-projection
reconstruction_fbp = iradon(sinogram_la, theta=theta, filter_name='ramp', circle=False)

# Calculation of the error
rmse_fbp = RMSE(image, reconstruction_fbp)
fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
axes[0].imshow(image, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('The original image')
axes[0].axis('off')
axes[1].imshow(reconstruction_fbp, cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'FBP reconstruction with limited angle\nRMSE = {rmse_fbp:.4g}')
axes[1].axis('off')
axes[2].imshow(reconstruction_fbp - image, cmap='gray')
axes[2].set_title('Error')
axes[2].axis('off')
plt.suptitle('Limited Angle Tomography Reconstruction', fontsize=11)
plt.tight_layout()
plt.show()

## Noise Model

**Noise** is any unwanted change in pixel values in an otherwise homogeneous image.

Here we consider the CT sinogram noise model **Additive White Gaussian Noise (AWGN)**:

$$p_{noise}(s,\theta) \;=\; p(s,\theta) \;+\; \eta(s,\theta),\qquad \eta(s,\theta) \overset{\text{i.i.d.}}{\sim}  \mathcal{N}(0,\sigma^2)$$

where $p(s,\theta)$ is the sinogram without noise, $\eta(s,\theta)$ is the noise modelled as a zero-mean normal distribution with variance $\sigma^2$.

The filters implemented in the `iradon`function can help us reducing the noise.

In the following cell, we create an array with different noise levels (which are the values of the standard deviation) and adopt two filters in the frequency domain:
- Ramp: $W(\omega) = |\omega|$
- Hamming: $W(\omega) = |\omega| (0.54 + 0.46\cos(\pi\omega/\omega_{max}))$

What is the difference between the two? The Hamming filter acts as a windowed version of the ramp filter that attenuates high frequencies.

Let us create a function that adds the AWGN to the sinogram.

In [ ]:
def add_awgn_to_sinogram(sinogram, sigma):
    s_noisy = sinogram + np.random.normal(0, sigma, sinogram.shape) # sigma = standard deviation
    # alternative with a built-in function
    # s_noisy = random_noise(sinogram, mode='gaussian', var=sigma**2, clip=False)
    return s_noisy

In the following code we will investigate the effect of noise

In [ ]:
# Define again the phantom and compute the noise-free sinogram
image = shepp_logan_phantom()
image = rescale(image, scale=0.4, mode='reflect', channel_axis=None)
theta = np.linspace(0., 180., max(image.shape), endpoint=False)
sinogram = radon(image,theta=theta)

# Add different levels of noise to the sinogram
noise_levels = [0.1,1,3,6]
fig, axes = plt.subplots(len(noise_levels), 3, figsize=(18, 5*len(noise_levels)))

for i, sigma in enumerate(noise_levels):
    # Add Gaussian noise to sinogram using the function random_noise
    s_noisy = add_awgn_to_sinogram(sinogram,sigma)
    
    # Reconstruct with ramp filter (sensitive to noise)
    reconstruction_ramp = iradon(s_noisy, theta=theta, 
                                 filter_name='ramp', circle=True)
    e_ramp = RMSE(image, reconstruction_ramp)
    
    # Reconstruct with Hamming filter (noise-robust)
    reconstruction_hamming = iradon(s_noisy, theta=theta, 
                                    filter_name='hamming', circle=True)
    e_hamming = RMSE(image,reconstruction_hamming)
    
    # Plot noisy sinogram
    axes[i, 0].imshow(s_noisy, cmap='gray', aspect='auto')
    axes[i, 0].set_title(f'Noisy Sinogram (σ={sigma:.2f})', fontsize=12)
    axes[i, 0].axis('off')
    
    # Plot ramp reconstruction
    axes[i, 1].imshow(reconstruction_ramp, cmap='gray')
    axes[i, 1].set_title(f'Ramp Filter\nRMSE: {e_ramp:.4f}', fontsize=12)
    axes[i, 1].axis('off')
    
    # Plot Hamming reconstruction
    axes[i, 2].imshow(reconstruction_hamming, cmap='gray')
    axes[i, 2].set_title(f'Hamming Filter\nRMSE: {e_hamming:.4f}', fontsize=12)
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()

**Questions**: 
1. How does noise change in the different reconstructions when changing the noise level?
2. How do filters affect the reconstructions? Why?

Discuss for 2 minutes with your neighbour about these questions.

## 3D Phantom Construction

We construct a 3D phantom that consists of the following parts:
- an ellipsoid with attenuation value 1, parametrized as
$$\frac{x^2}{a^2} + \frac{y^2}{b^2} + \frac{z^2}{c^2} <1$$
- a cylindrical structure with base in the centre of the `xy` plane that runs along all slices with attenuation value 0.3
- a small sphere shifted from the center of the `xy`-plane, centred in the middle of the z-axis with attenuation value 0.6

The phantom is a 3D array of shape `(z-slices, height, width)`.

In [ ]:
# initialize our empty 3D array
z_slices = 64
height = 128
width = 128
phantom3d = np.zeros((z_slices, height, width))
center_x = width // 2
center_y = height // 2
center_z = z_slices //2
# parameters ellipsoid
a, b, c = 40, 50, 20
# radius cylinder
radius_cyl = 8
# Shift sphere
shift = 20
# radius sphere
radius_sphere = 12
# Attenuation coefficients
attenuation_coeff = [1,0.3,0.6]



for z in range(z_slices):
    
    # initialize the ellipsoid
    
    for i in range(width):
        for j in range(height):
            if ((i-center_x)**2/a**2 + (j-center_y)**2/b**2 + (z-center_z)**2/c**2 < 1):
                phantom3d[z,j,i] = attenuation_coeff[0]

    # Cylinder — same cross-section for every z
    for i in range(width):
        for j in range(height):
            if ((i - center_x)**2 + (j - center_y)**2<radius_cyl**2):
                phantom3d[z][j,i] = attenuation_coeff[1]
    # Small sphere
    for i in range(width):
        for j in range(height):
            if ((i - center_x -shift)**2 + (j - center_y -shift)**2 + (z- center_z)**2<radius_sphere**2):
                phantom3d[z][j,i] = attenuation_coeff[2]

In [ ]:
# Vectorized version
# Instead of loops, use masks!
x_grid = np.arange(width)
y_grid = np.arange(height)
z_grid = np.arange(z_slices)

Z, Y, X = np.meshgrid(z_grid, y_grid, x_grid, indexing='ij')

# mask ellipsoid
mask_ellipsoid = ((X - center_x)**2 / a**2 +(Y - center_y)**2 / b**2 +
    (Z - center_z)**2 / c**2 < 1)

# mask cylinder
mask_cylinder = ((X - center_x)**2 + (Y - center_y)**2 < radius_cyl**2)

# small shifted sphere mask
mask_sphere = ((X - center_x-shift)**2 + (Y - center_y -shift)**2 +
               (Z-center_z)**2 < radius_sphere**2)

# Application of the masks to the 3d phantom
phantom_3d = np.zeros((z_slices, height, width))
phantom_3d[mask_ellipsoid] = attenuation_coeff[0]
phantom_3d[mask_cylinder]  = attenuation_coeff[1]
phantom_3d[mask_sphere]    = attenuation_coeff[2]

In [ ]:
# Print information about the 3D-array
print(f'  Shape                : {phantom_3d.shape}  -> (z-slices, height, width)')
print(f'  Each slice           : {phantom_3d.shape[1]} x {phantom_3d.shape[2]} pixels')
print(f'  Attenuation range    : [{phantom_3d.min():.2f},  {phantom_3d.max():.2f}]')
print(f'  Total voxels         : {phantom_3d.size:,}')

In [ ]:
plt.title(f'Original Phantom Slice {z_slices // 2}')
plt.imshow(phantom_3d[z_slices // 2], cmap='gray')
plt.axis('off')
plt.show()

Let us visualize our phantom through the `z` slices. We use a function that comes from the Archeolab.

In [ ]:
def show_slice(rec):
    # Ensure rec is a 3D numpy array
    assert rec.ndim == 3, "Input data must be a 3D numpy array"
    
    # Get the dimensions of the 3D data
    z_dim, y_dim, x_dim = rec.shape
    
    # Create an image widget with a height based on the z-dimension of the data
    image_widget = Image(height=z_dim)
    
    def update_slice(change):
        # Extract the slice at the given z-index
        z_index = change['new']
        slice_data = rec[z_index, :,:]
        
        # Normalize the slice data to the range [0, 255]
        slice_data_normalized = ((slice_data - slice_data.min()) / (slice_data.max() - slice_data.min()) * 255).astype(np.uint8)
        
        # Convert the numpy array to a PIL image
        pil_image = PILImage.fromarray(slice_data_normalized)
        
        # Save the PIL image to a buffer
        buf = io.BytesIO()
        pil_image.save(buf, format='png')
        
        # Update the image widget with the new image data
        buf.seek(0)
        image_widget.value = buf.read()

    
    # Create a slider for selecting the x-index
    z_slider = IntSlider(min=0, max=z_dim-1, step=1, description='Z-index')
    
    # Link the slider to the update_slice function
    z_slider.observe(update_slice, names='value')
    
    update_slice({'new': z_slider.value})

    # Display the slider and image widget in a vertical box with fixed height
    display(HBox([z_slider, image_widget], layout=Layout(height=f'256px')))


In [ ]:
show_slice(phantom_3d)

### Exercise 2
Create a function called `create_3d_phantom` that implements a sphere in the center of the volume following the structure of the above code.

As **input** it takes the following variables:
- `z_slices`, `height`, `width` corresponding to the shape of the 3d phantom.
- `radius`which corresponds to the radius of the sphere.
- `attenuation_coeff` corresponding to the attenuation coefficient of the sphere.

As **output**, it returns a three dimensional array.

Test your code and plot the central slice using `z_slices // 2`.

No error handling is required.

In [ ]:
# Write your function here
def create_3d_phantom(z_slices, height, width, radius, attenuation_coeff):
    pass

In [ ]:
# Test your code here

---
## 3D CT: Build the Sinogram Stack

### The 3D reconstruction strategy (parallel-beam)

For **parallel-beam** geometry, the 3D problem factorises into independent 2D problems — one per axial slice.

**Algorithm:**
1. For each slice $z$: apply 2D `radon` → sinogram of shape `(detectors, angles)`.
2. Stack → 3D sinogram of shape `(num_slices, detectors, angles)`.
3. Apply `iradon` independently to each slice → 3D reconstruction.

In [ ]:
# Parameters
num_angles = max(phantom_3d.shape)
num_detectors = max(phantom_3d.shape)

theta = np.linspace(0, 180, num_angles, endpoint=False)

sinograms = np.zeros((z_slices, num_detectors, num_angles), dtype=np.float32)
for z in range(z_slices):
    sinograms[z] = radon(phantom_3d[z], theta=theta)

print(f'Sinogram stack shape: {sinograms.shape}')

In [ ]:
def fbp_3d(sinograms, angles_deg, output_shape):
    """
    3D FBP reconstruction using skimage.iradon slice by slice.
    
    sinograms:   shape (num_slices, num_detectors, num_angles) — skimage convention
    angles_deg:  1D array of angles in degrees
    output_shape: (z, y, x)
    """
    num_slices = sinograms.shape[0]
    recon = np.zeros(output_shape, dtype=np.float32)
    
    for z in range(num_slices):
        recon[z] = iradon(sinograms[z], theta=angles_deg, filter_name='ramp')
    
    return recon

In [ ]:
output_shape = (z_slices, height, width)
# Reconstruction and plot a cross-section
recon = fbp_3d(sinograms, theta, output_shape)
print(f'Reconstruction shape: {recon.shape}')

plt.figure(figsize=(12, 4))
plt.subplot(1, 3, 1)
plt.title(f'Original Phantom Slice {z_slices // 2}')
plt.imshow(phantom_3d[z_slices // 2], cmap='gray')
plt.axis('off')

plt.subplot(1, 3, 2)
plt.title(f'Sinogram of the Slice {z_slices // 2}')
plt.imshow(sinograms[z_slices // 2,:, :], cmap='gray', aspect='auto')
plt.axis('off')

plt.subplot(1, 3, 3)
plt.title(f'Reconstruction')
plt.imshow(recon[z_slices // 2], cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
show_slice(recon)

# End for Today!

# Programming Exercise
The first exercise is mandatory and has to be submitted with the theoretical exercise with deadline after Easter holidays. The other one is optional but strongly recommended if you are interested in improving your programming skills.

## Exercise 1 (Obligatory): Noise + Sparse angle tomography

In this exercise we will experiment the FBP reconstruction in the case of undersampling and the presence of noise.

Take the Shepp Logan phantom as the original image. Consider a 1D array of projections angles called `num_angles` with values `[30, 60, 120, 240]`. Initialize two empty lists named `rmse_shepp` and `rmse_cosine`.

Create a `for` loop that loops over the number of projection angles and performs the following steps.
- Initialize the array of projection angles `theta`.
- Create the sinogram.
- Create a noisy sinogram using the `add_awgn_to_sinogram` function with noise level $\sigma=2$.
- Reconstruct the image using two filters, the `shepp-logan` and the `cosine` filter.
- Compute the value of the RMSE for the different filters and append them to the lists defined above.
- Plot the three pictures: the original phantom, the reconstruction with the Shepp-Logan filter with the RMSE in the title, the reconstruction with the Cosine filter with the RMSE in the title.
- Plot the RMSE versus the projection angles.

**Questions**
- Does increasing the number of projection angles improve the reconstruction when the data are noisy?
- At what point does noise dominate over the sampling angle error?

In [ ]:
# Write your code here
# num_angles = [30, 60, 120, 240]

## Exercise 2 (Optional)

Define a 3d phantom using the function introduced during the course. You can initialize it with values `z_slices = 64`, `height = 128`, `width = 128`, `radius = 50`, `attenuation_coeff = 1`.

Following the code introduced during the lecture, define two stacks of noisy sinograms, one with level $\sigma=0.5$ and the other with level $\sigma=2$. 

Apply the function `fbp_3d` to the 3d phantom to reconstruct it in both cases. 
Compute and print the RMSE from the slice 30 to 40.

**Question**
- Is the error uniform across the slices? Why?

In [ ]:
# Write your code here

## References
- PhD Thesis "Reconstructions in limited angle x-ray tomography: Characterization of classical reconstructions and adapted curvelet sparse regularization", Jürgen Frikel (https://mediatum.ub.tum.de/doc/1115037/287292.pdf)
- https://radiopaedia.org/articles/noise-ct
- Kak and Slaney, "Principles of Computerized Tomographic Imaging" (2001)